# 10b_enterocytes — Deep-Dive Differential Accessibility in Enterocytes

**Goal:** Full multi-level differential accessibility analysis focused exclusively on
**Enterocytes** — the major absorptive epithelial cell type of the small intestine.

## Analysis levels

### Level 1 — Focused DESeq2
- Re-run shared-peaks and evolutionary branch analysis on enterocytes only
- Higher stringency than all-cell-type run: `min_samples = 3`, `sd_thresh = 2.5`
- Per-contrast outlier detection + named outlier log
- Ultra-robust filtering (`min(pos) > max(neg)` in logCPM)
- Volcano plots per contrast, species marker heatmap, evolutionary staircase

### Level 2 — Motif Enrichment
- monaLisa (JASPAR2022 vertebrate PWMs) on peaks from each phylogenetic node
- Separate enrichment for: phylogenetic nodes, ILS contrasts, human-specific peaks
- Motif enrichment heatmap

### Level 3 — Gene Annotation, GO Enrichment & Visualization
- Nearest-gene annotation for all significant peaks
- GO enrichment (BP) per evolutionary contrast
- Accessibility dotplots (per-species) for top peaks at each node
- Individual strip plots for hand-picked top loci
- Per-region (Duodenum vs Colon) comparison using region-preserved data

**Requires:**
- `pb_data_shared.rds` / `pb_data_union.rds` from `10a`
- `pb_data_shared_per_region.rds` / `pb_data_union_per_region.rds` from `10a` (Cell 10)
- `master_annotation_final.tsv`

**Outputs:**
- `differential_results/enterocytes/` — per-contrast CSV + ultra-robust CSV
- `plots/Enterocytes_Deep/` — all plots for this analysis

In [1]:
# =============================================================================
# Cell 1: Load Packages & Source Utilities
# =============================================================================
suppressPackageStartupMessages({
  library(DESeq2)
  library(ggplot2)
  library(ggrepel)
  library(pheatmap)
  library(viridis)
  library(ComplexHeatmap)
  library(circlize)
  library(matrixStats)
  library(dplyr)
  library(tibble)
  library(tidyr)
  library(readr)
})

source("../src/deseq2_utils.R")
message("Packages loaded & utilities sourced.")

Packages loaded & utilities sourced.



In [2]:
# =============================================================================
# Cell 2: Configuration & Data Loading
# =============================================================================
BASE    <- "/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks"
OUT_DIR <- file.path(BASE, "cross_species_consensus_v3/13_deseq2_R_pseudobulk")
SPECIES <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")

ENT_OUT  <- file.path(OUT_DIR, "differential_results", "enterocytes")
ENT_PLOT <- file.path(OUT_DIR, "plots", "Enterocytes_Deep")
dir.create(ENT_OUT,  showWarnings = FALSE, recursive = TRUE)
dir.create(ENT_PLOT, showWarnings = FALSE, recursive = TRUE)

save_dir <- file.path(OUT_DIR, "pseudobulk")

# --- Load global pseudobulk data (aggregated, 241 samples) ---
pb_shared    <- readRDS(file.path(save_dir, "pb_data_shared.rds"))
pb_union     <- readRDS(file.path(save_dir, "pb_data_union.rds"))
counts_all   <- pb_shared$counts
meta_all     <- pb_shared$meta
counts_union <- pb_union$counts

# --- Load per-region data for Level 3 enterocyte region analysis ---
pb_shared_reg <- readRDS(file.path(save_dir, "pb_data_shared_per_region.rds"))
pb_union_reg  <- readRDS(file.path(save_dir, "pb_data_union_per_region.rds"))

# --- Subset to enterocytes ---
# The cell type name after make.names() is "Enterocytes"
ENT_CT <- "Enterocytes"
meta_ent   <- meta_all[meta_all$cell_type == ENT_CT, ]
counts_ent <- counts_all[, rownames(meta_ent)]

message("Enterocyte samples: ", nrow(meta_ent))
print(table(meta_ent$species))

# --- Load master annotation & build orthology index ---
anno_file   <- file.path(BASE, "cross_species_consensus_v3/07_master_annotation/master_annotation_final.tsv")
master_anno <- read_tsv(anno_file, show_col_types = FALSE)
options(scipen = 999)

ortho_mat <- build_orthology_index(master_anno, SPECIES)
message("Orthology index: ", nrow(ortho_mat), " peaks x ", ncol(ortho_mat), " species")

Enterocyte samples: 11




     Human     Bonobo Chimpanzee    Gorilla    Macaque   Marmoset 
         4          2          1          1          1          2 


Orthology index: 1142003 peaks x 6 species



Orthology index: 1142003 peaks x 6 species



---
## Level 1: Focused DESeq2 on Enterocytes

Higher stringency than the all-cell-type run: `min_samples = 3` (enterocytes are
well-represented, so this is achievable) and `sd_thresh = 2.5` for outlier detection.

In [3]:
# =============================================================================
# Cell 3: Shared-Peak Analysis (enterocytes, all-vs-one)
# =============================================================================
# Build a meta_shared-compatible object with only enterocytes
meta_ent_factor           <- meta_ent
meta_ent_factor$cell_type <- factor(ENT_CT)
meta_ent_factor$species   <- factor(meta_ent_factor$species, levels = SPECIES)

# Re-run with higher stringency: min 3 samples per peak
res_ent_shared <- run_deseq2_shared_peaks(
  counts_shared   = counts_ent,
  meta_shared     = meta_ent_factor,
  species         = SPECIES,
  out_dir         = ENT_OUT,
  filter_outliers = TRUE,
  sd_thresh       = 2.5
)

# Summary
for (sp in names(res_ent_shared[[ENT_CT]])) {
  res_df  <- as.data.frame(res_ent_shared[[ENT_CT]][[sp]])
  sig_up  <- sum(!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange > 1)
  sig_dn  <- sum(!is.na(res_df$padj) & res_df$padj < 0.05 & res_df$log2FoldChange < -1)
  message(sprintf("  %-12s: %d UP, %d DOWN", sp, sig_up, sig_dn))
}


=== [Shared Peaks] Enterocytes ===



converting counts to integer mode



  Marmoset     specific: 6010 (padj<0.05, LFC>1)



  Human        specific: 5624 (padj<0.05, LFC>1)



  Chimpanzee   specific: 39 (padj<0.05, LFC>1)



...

  Human       : 5624 UP, 2365 DOWN



  Chimpanzee  : 39 UP, 306 DOWN



  Bonobo      : 52 UP, 972 DOWN



  Gorilla     : 1288 UP, 138 DOWN



  Macaque     : 3123 UP, 1643 DOWN



In [4]:
# =============================================================================
# Cell 4: Evolutionary Branch Analysis (enterocytes)
# =============================================================================
evo_contrasts <- define_evolutionary_contrasts()

branch_ent <- run_deseq2_evolutionary(
  counts_union    = counts_union,
  meta_shared     = meta_ent_factor,
  contrasts       = evo_contrasts,
  ortho_mat       = ortho_mat,
  out_dir         = ENT_OUT,
  min_samples     = 3,
  filter_outliers = TRUE,
  sd_thresh       = 2.5
)

message("\nEvolutionary branch analysis complete for enterocytes.")
message("Contrasts with results: ", length(branch_ent[[ENT_CT]]))

Defined 19 evolutionary contrasts.




=== [Evolutionary Branches] Enterocytes ===



converting counts to integer mode



  Node1_Human_vs_Pan                      : 110762 peaks tested, 11926 sig UP



converting counts to integer mode



...

  Node_Apes_vs_Monkeys                    : 207034 peaks tested, 14268 sig UP




Outlier filtering: no samples removed across all contrasts.




Checkpoint: evolutionary branch results saved.




Evolutionary branch analysis complete for enterocytes.



Contrasts with results: 19



In [5]:
# =============================================================================
# Cell 5: Ultra-Robust Filtering
# =============================================================================
robust_ent <- ultra_robust_filter(
  branch_res   = branch_ent,
  counts_union = counts_union,
  meta_shared  = meta_ent_factor,
  contrasts    = evo_contrasts,
  out_dir      = ENT_OUT,
  padj_thresh  = 0.05,
  lfc_thresh   = 1,
  min_logcpm   = 1
)

# Summarise ultra-robust peak counts per contrast
ur_counts <- sapply(robust_ent[[ENT_CT]], length)
ur_df     <- data.frame(
  contrast = names(ur_counts),
  n_ultra_robust = as.integer(ur_counts)
) %>% arrange(desc(n_ultra_robust))

message("\nUltra-robust peaks per contrast (enterocytes):")
print(ur_df)


Ultra-robust filtering: Enterocytes



  [Node1_Human_vs_Pan] DESeq2 UP: 11926 <e2><86><92> Ultra-Robust: 9135



  [Node2_AfricanApes_vs_Gorilla] DESeq2 UP: 1190 <e2><86><92> Ultra-Robust: 1066



  [Node3_GreatApes_vs_Macaque] DESeq2 UP: 6399 <e2><86><92> Ultra-Robust: 3906



  [Node4_OldWorld_vs_Marmoset] DESeq2 UP: 17284 <e2><86><92> Ultra-Robust: 4787



...

  [Div_Human_vs_AllPrimates] DESeq2 UP: 24697 <e2><86><92> Ultra-Robust: 6496



  [Node_Apes_vs_Monkeys] DESeq2 UP: 14268 <e2><86><92> Ultra-Robust: 4048




Ultra-robust filtering complete. Saved checkpoint.




Ultra-robust peaks per contrast (enterocytes):



                          contrast n_ultra_robust
1           Pair_Human_vs_Marmoset          20168
2            Pair_Human_vs_Macaque          13780
3               Node1_Human_vs_Pan           9135
4              Pair_Human_vs_Chimp           9031
5                Div_Human_vs_Apes           7774
6         Div_Human_vs_AllPrimates           6496
7         Div_Macaque_vs_GreatApes           5851
8       Node4_OldWorld_vs_Marmoset           4787
9             Node_Apes_vs_Monkeys           4048
10           Pair_Human_vs_Gorilla           3957
11      Node3_GreatApes_vs_Macaque           3906
12         ILS_HumanGorilla_vs_Pan           3885
13            Pair_Human_vs_Bonobo           3165
14             Div_Gorilla_vs_Apes           2483
15    Node2_AfricanApes_vs_Gorilla           1066
16 ILS_HumanChimp_vs_GorillaBonobo            884
17 ILS_HumanBonobo_vs_ChimpGorilla            764
18              Div_Bonobo_vs_Apes            262
19               Div_Chimp_vs_Apes            117


In [6]:
# =============================================================================
# Cell 6: Volcano Plots (shared-peaks + key evolutionary branches)
# =============================================================================

# A) Shared-peak volcanos (one per species)
for (sp in names(res_ent_shared[[ENT_CT]])) {
  res_df   <- as.data.frame(res_ent_shared[[ENT_CT]][[sp]])
  out_file <- file.path(ENT_PLOT, paste0("Volcano_SharedPeaks_", sp, ".pdf"))
  plot_volcano(res_df,
               title    = paste(sp, "vs All Others — Enterocytes [shared peaks]"),
               out_file = out_file, n_label = 12)
}

# B) Key evolutionary branch volcanos
key_contrasts <- c(
  "Node1_Human_vs_Pan", "Node2_AfricanApes_vs_Gorilla",
  "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys",
  "Div_Human_vs_Apes", "Div_Human_vs_AllPrimates",
  "ILS_HumanGorilla_vs_Pan", "ILS_HumanChimp_vs_GorillaBonobo",
  "Pair_Human_vs_Macaque", "Pair_Human_vs_Marmoset"
)

for (cn in intersect(key_contrasts, names(branch_ent[[ENT_CT]]))) {
  res_df   <- as.data.frame(branch_ent[[ENT_CT]][[cn]])
  out_file <- file.path(ENT_PLOT, paste0("Volcano_Branch_", cn, ".pdf"))
  plot_volcano(res_df,
               title    = paste(cn, "— Enterocytes [evolutionary branch]"),
               out_file = out_file, n_label = 12)
}

message("All volcano plots saved.")

  Volcano saved: Volcano_SharedPeaks_Marmoset.pdf



  Volcano saved: Volcano_SharedPeaks_Human.pdf



  Volcano saved: Volcano_SharedPeaks_Chimpanzee.pdf



  Volcano saved: Volcano_SharedPeaks_Bonobo.pdf



  Volcano saved: Volcano_SharedPeaks_Gorilla.pdf



...

  Volcano saved: Volcano_Branch_ILS_HumanGorilla_vs_Pan.pdf



  Volcano saved: Volcano_Branch_ILS_HumanChimp_vs_GorillaBonobo.pdf



  Volcano saved: Volcano_Branch_Pair_Human_vs_Macaque.pdf



  Volcano saved: Volcano_Branch_Pair_Human_vs_Marmoset.pdf



All volcano plots saved.



In [7]:
# =============================================================================
# Cell 7: Species Marker Heatmap (shared-peak results)
# =============================================================================
# Build a temporary VST for enterocytes only (more sensitive than global VST)
dds_ent_vst <- DESeqDataSetFromMatrix(
  countData = round(counts_ent),
  colData   = meta_ent_factor,
  design    = ~ species
)
dds_ent_vst <- estimateSizeFactors(dds_ent_vst, type = "poscounts")
vsd_ent     <- vst(dds_ent_vst, blind = FALSE)
message("Enterocyte-specific VST computed.")

plot_species_heatmap(
  res_ct    = res_ent_shared[[ENT_CT]],
  vsd       = vsd_ent,
  meta      = meta_ent_factor,
  cell_type = ENT_CT,
  out_file  = file.path(ENT_PLOT, "Heatmap_Species_Markers.pdf"),
  top_n     = 30
)

converting counts to integer mode



Enterocyte-specific VST computed.



  Marmoset: 6010 sig peaks, top 30 selected



  Human: 5624 sig peaks, top 30 selected



  Chimpanzee: 39 sig peaks, top 30 selected



...

  Gorilla: 1288 sig peaks, top 30 selected



  Macaque: 3123 sig peaks, top 30 selected



  Total unique top regions: 175



  After VST intersect: 175 regions



  Heatmap saved: Heatmap_Species_Markers.pdf



In [8]:
# =============================================================================
# Cell 8: Evolutionary Staircase Heatmap
# =============================================================================
species_order <- c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset")
node_order    <- c(
  "Node1_Human_vs_Pan", "Node2_AfricanApes_vs_Gorilla",
  "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys", "Div_Human_vs_Apes",
  "Div_Human_vs_AllPrimates"
)

ct_nodes <- intersect(node_order, names(robust_ent[[ENT_CT]]))
ct_nodes <- ct_nodes[sapply(ct_nodes,
  function(n) length(robust_ent[[ENT_CT]][[n]]) > 0)]

# Compute logCPM averages per species
ent_samples <- rownames(meta_ent)
ent_counts_u <- counts_union[, ent_samples, drop = FALSE]
lib_sizes  <- colSums(ent_counts_u)
cpm_mat    <- t(t(ent_counts_u) / lib_sizes) * 1e6
logcpm_ent <- log2(cpm_mat + 1)

avg_exp <- matrix(NA, nrow = nrow(logcpm_ent), ncol = length(species_order),
                  dimnames = list(rownames(logcpm_ent), species_order))
for (sp in species_order) {
  sp_samp <- ent_samples[meta_ent$species == sp]
  if (length(sp_samp) > 1)
    avg_exp[, sp] <- rowMeans(logcpm_ent[, sp_samp, drop = FALSE])
  else if (length(sp_samp) == 1)
    avg_exp[, sp] <- logcpm_ent[, sp_samp]
}

plot_peaks <- c()
row_splits <- c()
for (node in ct_nodes) {
  ur_peaks <- robust_ent[[ENT_CT]][[node]]
  if (length(ur_peaks) == 0) next
  res_df   <- as.data.frame(branch_ent[[ENT_CT]][[node]])
  ur_res   <- res_df[intersect(ur_peaks, rownames(res_df)), , drop = FALSE]
  sorted   <- rownames(ur_res)[order(ur_res$log2FoldChange, decreasing = TRUE)]
  top      <- head(sorted, 50)
  plot_peaks <- c(plot_peaks, top)
  row_splits <- c(row_splits, rep(node, length(top)))
}

if (length(plot_peaks) >= 5) {
  mat <- avg_exp[plot_peaks, , drop = FALSE]
  valid_cols <- apply(mat, 2, function(x) !all(is.na(x)))
  mat <- mat[, valid_cols, drop = FALSE]
  mat_scaled <- t(apply(mat, 1, scale))
  colnames(mat_scaled) <- colnames(mat)
  split_factor <- factor(row_splits, levels = ct_nodes)

  col_fun <- colorRamp2(c(-2, 0, 2), c("#377eb8", "white", "#e41a1c"))
  ht <- Heatmap(mat_scaled, name = "Z-score", col = col_fun,
                row_split = split_factor, row_title_rot = 0,
                row_title_gp = gpar(fontsize = 10, fontface = "bold"),
                row_gap = unit(3, "mm"), cluster_rows = TRUE,
                cluster_columns = FALSE, show_row_names = FALSE,
                show_column_names = TRUE, column_names_rot = 45,
                column_title = "Evolutionary Staircase — Enterocytes",
                column_title_gp = gpar(fontsize = 14, fontface = "bold"),
                heatmap_legend_param = list(title = "Z-score"))

  dynamic_h <- max(10, length(plot_peaks) * 0.09 + length(ct_nodes))
  pdf(file.path(ENT_PLOT, "Evolutionary_Staircase_Enterocytes.pdf"),
      width = 10, height = dynamic_h)
  draw(ht, merge_legend = TRUE)
  dev.off()
  message("Staircase heatmap saved (", length(plot_peaks), " peaks).")
}

Staircase heatmap saved (350 peaks).



---
## Level 2: Motif Enrichment

Run monaLisa binned motif enrichment (JASPAR2022 vertebrate PWMs) on peak groups
defined by phylogenetic node. Each group = ultra-robust peaks for one contrast.

**Groups tested:**
1. Phylogenetic nodes (Node1–4 + Node_Apes_vs_Monkeys)
2. ILS contrasts (Human+Gorilla vs Pan; Human+Chimp vs Gorilla+Bonobo)
3. Human-specific (Div_Human_vs_Apes + Div_Human_vs_AllPrimates)

Background = all enterocyte accessible peaks (shared, 71k).

In [9]:
# =============================================================================
# Cell 9: Motif Enrichment — Phylogenetic Node Peaks
# =============================================================================
suppressPackageStartupMessages({
  library(monaLisa)
  library(JASPAR2022)
  library(TFBSTools)
  library(BSgenome.Hsapiens.UCSC.hg38)
  library(Biostrings)
})

motif_dir <- file.path(ENT_OUT, "motif_enrichment")
dir.create(motif_dir, showWarnings = FALSE, recursive = TRUE)

# --- Build peak groups for motif enrichment ---
# Use ultra-robust peaks; label each by the contrast it came from.
# Only keep groups with >= 20 peaks (minimum for monaLisa to be stable).
motif_groups <- c(
  # Phylogenetic nodes
  "Node1_Human_vs_Pan", "Node2_AfricanApes_vs_Gorilla",
  "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys",
  # ILS
  "ILS_HumanGorilla_vs_Pan", "ILS_HumanChimp_vs_GorillaBonobo",
  # Human-specific
  "Div_Human_vs_Apes", "Div_Human_vs_AllPrimates"
)

peak_list_motif <- list()
for (cn in intersect(motif_groups, names(robust_ent[[ENT_CT]]))) {
  peaks <- robust_ent[[ENT_CT]][[cn]]
  if (length(peaks) >= 20) {
    peak_list_motif[[cn]] <- peaks
    message(sprintf("  %-40s: %d peaks", cn, length(peaks)))
  } else {
    message(sprintf("  %-40s: %d peaks — SKIPPED (< 20)", cn, length(peaks)))
  }
}

if (length(peak_list_motif) == 0) {
  message("No groups with >= 20 peaks. Adjust thresholds or run DESeq2 with looser settings.")
} else {
  message("Running motif enrichment for ", length(peak_list_motif), " groups...")

  se_motif_nodes <- run_parallel_motif_enrichment(
    peak_list_named = peak_list_motif,
    anno_df         = master_anno,
    species         = "Human",
    n_cores         = 4L,
    out_rds         = file.path(motif_dir, "se_motif_phylo_nodes.rds")
  )

  plot_motif_heatmap(
    se_obj      = se_motif_nodes,
    title       = "Motif Enrichment — Enterocyte Phylogenetic Node Peaks",
    out_file    = file.path(ENT_PLOT, "Motif_Heatmap_PhyloNodes.pdf"),
    top_per_col = 5
  )
  message("Phylogenetic node motif enrichment complete.")
}

  Node1_Human_vs_Pan                      : 9135 peaks



  Node2_AfricanApes_vs_Gorilla            : 1066 peaks



  Node3_GreatApes_vs_Macaque              : 3906 peaks



  Node4_OldWorld_vs_Marmoset              : 4787 peaks



  Node_Apes_vs_Monkeys                    : 4048 peaks



...

Warning message:
"<anonymous>: ... may be used in an incorrect context: 
     findMotifHits(query = pwmL, subject = df$seqs[!df$isForeground], 
         min.score = min.score, method = matchMethod, BPPARAM = BPPARAM, 
         ...)
"


Combining 1 batch results...



Saved SE object: se_motif_phylo_nodes.rds



  Motif heatmap saved: Motif_Heatmap_PhyloNodes.pdf



Phylogenetic node motif enrichment complete.



In [10]:
# =============================================================================
# Cell 10: Motif Enrichment — ILS Contrast Peaks
# =============================================================================
ils_groups <- c(
  "ILS_HumanGorilla_vs_Pan",
  "ILS_HumanChimp_vs_GorillaBonobo",
  "ILS_HumanBonobo_vs_ChimpGorilla"
)

peak_list_ils <- list()
for (cn in intersect(ils_groups, names(robust_ent[[ENT_CT]]))) {
  peaks <- robust_ent[[ENT_CT]][[cn]]
  if (length(peaks) >= 20) peak_list_ils[[cn]] <- peaks
}

if (length(peak_list_ils) >= 2) {
  se_motif_ils <- run_parallel_motif_enrichment(
    peak_list_named = peak_list_ils,
    anno_df         = master_anno,
    species         = "Human",
    n_cores         = 4L,
    out_rds         = file.path(motif_dir, "se_motif_ILS.rds")
  )
  plot_motif_heatmap(
    se_obj      = se_motif_ils,
    title       = "Motif Enrichment — Enterocyte ILS Peaks",
    out_file    = file.path(ENT_PLOT, "Motif_Heatmap_ILS.pdf"),
    top_per_col = 5
  )
  message("ILS motif enrichment complete.")
} else {
  message("Insufficient ILS groups with >= 20 peaks. Skipping.")
}

Total groups: 3 <e2><80><94> processing in batches of 25



Peak memory per batch: ~18.4 MB sequences



Parallel backend: SnowParam (4 workers)



  Batch 1/1: 3 groups, 5533 sequences



Warning message:
"<anonymous>: ... may be used in an incorrect context: 
     findMotifHits(query = pwmL, subject = df$seqs[!df$isForeground], 
         min.score = min.score, method = matchMethod, BPPARAM = BPPARAM, 
         ...)
"


Combining 1 batch results...



Saved SE object: se_motif_ILS.rds



  Motif heatmap saved: Motif_Heatmap_ILS.pdf



ILS motif enrichment complete.



---
## Level 3: Gene Annotation, GO Enrichment & Visualization

In [11]:
# =============================================================================
# Cell 11: Gene Annotation for All Significant Peaks
# =============================================================================
gene_anno_dir <- file.path(ENT_OUT, "gene_annotation")
dir.create(gene_anno_dir, showWarnings = FALSE, recursive = TRUE)

# Collect all ultra-robust peaks with their contrast labels
all_ur_peaks <- lapply(names(robust_ent[[ENT_CT]]), function(cn) {
  peaks <- robust_ent[[ENT_CT]][[cn]]
  if (length(peaks) == 0) return(NULL)
  data.frame(peak_id = peaks, contrast = cn, stringsAsFactors = FALSE)
})
all_ur_df <- do.call(rbind, Filter(Negate(is.null), all_ur_peaks))

# Add gene annotation from master_anno
gene_info <- get_peak_info(all_ur_df$peak_id, "Human", "gene", master_anno)
dist_info <- get_peak_info(all_ur_df$peak_id, "Human", "distance", master_anno)

all_ur_annotated <- all_ur_df %>%
  left_join(gene_info,     by = "peak_id") %>%
  left_join(dist_info,     by = "peak_id") %>%
  left_join(
    do.call(rbind, lapply(names(branch_ent[[ENT_CT]]), function(cn) {
      res_df <- as.data.frame(branch_ent[[ENT_CT]][[cn]])
      data.frame(peak_id = rownames(res_df), contrast = cn,
                 log2FC = round(res_df$log2FoldChange, 3),
                 padj   = signif(res_df$padj, 4),
                 stringsAsFactors = FALSE)
    })),
    by = c("peak_id", "contrast")
  )

write.csv(all_ur_annotated,
          file.path(gene_anno_dir, "Enterocytes_UltraRobust_Annotated.csv"),
          row.names = FALSE)

message("Annotated ultra-robust peaks: ", nrow(all_ur_annotated))
message("Unique genes: ", length(unique(all_ur_annotated$gene[all_ur_annotated$gene != "."])))
head(all_ur_annotated[all_ur_annotated$contrast == "Div_Human_vs_Apes", ], 10)

Annotated ultra-robust peaks: 101559



Unique genes: 19282



,peak_id,contrast,gene,distance,log2FC,padj
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
24428,unified_605936,Div_Human_vs_Apes,RNU7-141P,25931,4.708,0.00000000000000002303
24429,unified_896049,Div_Human_vs_Apes,METTL27,19741,4.033,0.00000000000000002303
24430,unified_021031,Div_Human_vs_Apes,ENSG00000301774,24427,3.405,0.00000000000000002855
24431,unified_928009,Div_Human_vs_Apes,ENSG00000253932,61743,3.674,0.00000000000000003930
24432,unified_617363,Div_Human_vs_Apes,MX2,4609,4.938,0.00000000000000270500
24433,unified_000009,Div_Human_vs_Apes,MTATP6P1,79,11.048,0.00000000000000289000
24434,unified_401330,Div_Human_vs_Apes,MTCYBP13,1532,8.596,0.00000000000001202000
24435,unified_968352,Div_Human_vs_Apes,ADCK5,606,9.659,0.00000000000001745000
24436,unified_211375,Div_Human_vs_Apes,GPRC5D-AS1,0,5.732,0.00000000000006748000


In [12]:
# =============================================================================
# Cell 12: GO Enrichment per Evolutionary Contrast
# =============================================================================
# Run GO enrichment on nearest genes for each contrast's ultra-robust peaks
go_contrasts <- c(
  "Node1_Human_vs_Pan", "Node2_AfricanApes_vs_Gorilla",
  "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys",
  "Div_Human_vs_Apes", "Div_Human_vs_AllPrimates"
)

for (cn in intersect(go_contrasts, names(robust_ent[[ENT_CT]]))) {
  peaks <- robust_ent[[ENT_CT]][[cn]]
  if (length(peaks) < 10) next

  genes <- get_peak_info(peaks, "Human", "gene", master_anno)$gene
  genes <- unique(genes[genes != "." & !is.na(genes)])

  run_go_enrichment(
    genes   = genes,
    label   = paste0("Enterocytes_", cn),
    out_dir = ENT_OUT
  )
}

message("\nGO enrichment complete for all contrasts.")

  GO enrichment saved for: Enterocytes_Node1_Human_vs_Pan



  No significant GO terms for: Enterocytes_Node2_AfricanApes_vs_Gorilla



  No significant GO terms for: Enterocytes_Node3_GreatApes_vs_Macaque



  No significant GO terms for: Enterocytes_Node4_OldWorld_vs_Marmoset



  No significant GO terms for: Enterocytes_Node_Apes_vs_Monkeys



  GO enrichment saved for: Enterocytes_Div_Human_vs_Apes



  GO enrichment saved for: Enterocytes_Div_Human_vs_AllPrimates




GO enrichment complete for all contrasts.



In [13]:
# =============================================================================
# Cell 13: Accessibility Dotplots — Top Peaks per Node
# =============================================================================
# For each evolutionary contrast, plot top 40 ultra-robust peaks as a
# per-species dotplot (mean log2CPM, colored by accessibility level)

dotplot_contrasts <- c(
  "Node1_Human_vs_Pan", "Node2_AfricanApes_vs_Gorilla",
  "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys",
  "Div_Human_vs_Apes", "Div_Human_vs_AllPrimates",
  "ILS_HumanGorilla_vs_Pan", "ILS_HumanChimp_vs_GorillaBonobo"
)

ent_samples <- rownames(meta_ent)

for (cn in intersect(dotplot_contrasts, names(robust_ent[[ENT_CT]]))) {
  ur_peaks <- robust_ent[[ENT_CT]][[cn]]
  if (length(ur_peaks) == 0) next

  # Sort by LFC to show highest-contrast peaks first
  res_df   <- as.data.frame(branch_ent[[ENT_CT]][[cn]])
  ur_res   <- res_df[intersect(ur_peaks, rownames(res_df)), , drop = FALSE]
  sorted   <- rownames(ur_res)[order(ur_res$log2FoldChange, decreasing = TRUE)]
  top_peaks <- head(sorted, 40)

  tryCatch({
    plot_accessibility_dotplot(
      peak_ids     = top_peaks,
      counts       = counts_union[, ent_samples, drop = FALSE],
      meta         = meta_ent,
      out_file     = file.path(ENT_PLOT, paste0("Dotplot_", cn, ".pdf")),
      title        = paste("Enterocytes:", cn, "— Ultra-Robust Peaks"),
      max_peaks    = 40
    )
  }, error = function(e) message("  Dotplot failed for ", cn, ": ", e$message))
}

message("All accessibility dotplots saved.")

  Accessibility dotplot saved: Dotplot_Node1_Human_vs_Pan.pdf



  Accessibility dotplot saved: Dotplot_Node2_AfricanApes_vs_Gorilla.pdf



  Accessibility dotplot saved: Dotplot_Node3_GreatApes_vs_Macaque.pdf



  Accessibility dotplot saved: Dotplot_Node4_OldWorld_vs_Marmoset.pdf



  Accessibility dotplot saved: Dotplot_Node_Apes_vs_Monkeys.pdf



  Accessibility dotplot saved: Dotplot_Div_Human_vs_Apes.pdf



  Accessibility dotplot saved: Dotplot_Div_Human_vs_AllPrimates.pdf



  Accessibility dotplot saved: Dotplot_ILS_HumanGorilla_vs_Pan.pdf



  Accessibility dotplot saved: Dotplot_ILS_HumanChimp_vs_GorillaBonobo.pdf



All accessibility dotplots saved.



In [14]:
# =============================================================================
# Cell 14: Strip Plots — Individual Top Loci
# =============================================================================
# For each major contrast, make per-donor strip plots for the 5 highest-LFC
# ultra-robust peaks. Shows raw variability across donors — key for sanity-checking.

strip_contrasts <- c(
  "Div_Human_vs_Apes", "Div_Human_vs_AllPrimates",
  "Node1_Human_vs_Pan", "Node_Apes_vs_Monkeys"
)

strip_dir <- file.path(ENT_PLOT, "StripPlots_TopLoci")
dir.create(strip_dir, showWarnings = FALSE, recursive = TRUE)

for (cn in intersect(strip_contrasts, names(robust_ent[[ENT_CT]]))) {
  ur_peaks <- robust_ent[[ENT_CT]][[cn]]
  if (length(ur_peaks) == 0) next

  res_df   <- as.data.frame(branch_ent[[ENT_CT]][[cn]])
  ur_res   <- res_df[intersect(ur_peaks, rownames(res_df)), , drop = FALSE]
  top5     <- head(rownames(ur_res)[order(ur_res$log2FoldChange, decreasing = TRUE)], 5)

  # Also get gene annotation for labeling
  gene_lut <- setNames(
    get_peak_info(top5, "Human", "gene", master_anno)$gene,
    get_peak_info(top5, "Human", "gene", master_anno)$peak_id
  )

  for (pk in top5) {
    gene_label <- gene_lut[pk]
    title_str  <- if (!is.na(gene_label) && gene_label != ".") {
      paste0(pk, "  (", gene_label, ")  |", cn)
    } else {
      paste0(pk, "  |", cn)
    }
    out_file <- file.path(strip_dir,
                          paste0(cn, "_", gsub(":", "_", pk), ".pdf"))
    tryCatch(
      plot_peak_strip(pk,
                      counts   = counts_union[, ent_samples, drop = FALSE],
                      meta     = meta_ent,
                      out_file = out_file,
                      title    = title_str),
      error = function(e) message("  Strip plot failed: ", pk)
    )
  }
  message("  Strip plots saved for: ", cn)
}

message("\nAll strip plots saved.")

Warning message:
"The `fatten` argument of `geom_crossbar()` is deprecated as of ggplot2 4.0.0.
i Please use the `middle.linewidth` argument instead."


  Strip plot saved: Div_Human_vs_Apes_unified_000009.pdf



  Strip plot saved: Div_Human_vs_Apes_unified_968352.pdf



  Strip plot saved: Div_Human_vs_Apes_unified_000008.pdf



  Strip plot saved: Div_Human_vs_Apes_unified_401330.pdf



...

  Strip plot saved: Node_Apes_vs_Monkeys_unified_981615.pdf



  Strip plot saved: Node_Apes_vs_Monkeys_unified_243528.pdf



  Strip plot saved: Node_Apes_vs_Monkeys_unified_448094.pdf



  Strip plots saved for: Node_Apes_vs_Monkeys




All strip plots saved.



## Level 3b: Per-Region Analysis (Duodenum vs Colon)

Run the full evolutionary contrast set on enterocytes within Duodenum and Colon
separately, then compare. Enterocytes are more abundant in duodenum (absorptive);
expect more peaks there, but some regulatory elements may be colon-restricted.

In [15]:
# =============================================================================
# Cell 15: Per-Region DESeq2 for Enterocytes
# =============================================================================
TARGET_REGIONS <- c("Duodenum", "Colon")

meta_ent_reg    <- pb_shared_reg$meta[pb_shared_reg$meta$cell_type == ENT_CT, ]
counts_ent_u_reg <- pb_union_reg$counts[,
  intersect(rownames(meta_ent_reg), colnames(pb_union_reg$counts)),
  drop = FALSE]
meta_ent_reg <- meta_ent_reg[
  intersect(rownames(meta_ent_reg), colnames(counts_ent_u_reg)), ]

message("Per-region enterocyte samples:")
print(table(meta_ent_reg$region, meta_ent_reg$species))

region_res_ent <- run_deseq2_per_region(
  counts_union_regions  = counts_ent_u_reg,
  meta_regions          = meta_ent_reg,
  contrasts             = evo_contrasts,
  ortho_mat             = ortho_mat,
  out_dir               = ENT_OUT,
  target_regions        = TARGET_REGIONS,
  min_samples           = 2,
  min_cells_region      = 30,
  min_counts_region     = 20000,
  filter_outliers       = TRUE,
  sd_thresh             = 2.5
)

message("Per-region enterocyte DESeq2 complete.")

Per-region enterocyte samples:



          
           Human Bonobo Chimpanzee Gorilla Macaque Marmoset
  Colon        0      0          1       0       0        0
  Duodenum     4      2          1       2       1        2


Adult samples available: 13



Regions in data: Duodenum, Colon





========== REGION: Duodenum ==========



...

  [Duodenum | Enterocytes] Node_Apes_vs_Monkeys                    : 366146 peaks, 13703 sig UP





========== REGION: Colon ==========



  After QC: 1 samples (counts >= 20000, cells >= 30)




Per-region DESeq2 complete.



Per-region enterocyte DESeq2 complete.



In [16]:
# =============================================================================
# Cell 16: Region Comparison for Enterocytes
# =============================================================================
comp_dir_ent <- file.path(ENT_OUT, "region_comparison")
dir.create(comp_dir_ent, showWarnings = FALSE, recursive = TRUE)

key_contrasts_reg <- c(
  "Node1_Human_vs_Pan", "Div_Human_vs_Apes", "Div_Human_vs_AllPrimates",
  "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
  "Node_Apes_vs_Monkeys"
)

summary_reg_rows <- list()

for (cn in key_contrasts_reg) {
  comp <- compare_regions(region_res_ent, ENT_CT, cn,
                          padj_thresh = 0.05, lfc_thresh = 1)
  if (is.null(comp) || nrow(comp) == 0) {
    message("  No significant peaks for: ", cn)
    next
  }
  write.csv(comp, file.path(comp_dir_ent, paste0(cn, "_region_comparison.csv")),
            row.names = FALSE)

  n_both  <- sum(comp$category == "Both")
  n_duod  <- sum(comp$category == "Duodenum_only")
  n_colon <- sum(comp$category == "Colon_only")
  message(sprintf("  %-40s Both=%d  Duod=%d  Colon=%d",
                  cn, n_both, n_duod, n_colon))

  summary_reg_rows[[cn]] <- data.frame(
    contrast = cn, n_both = n_both,
    n_duodenum_only = n_duod, n_colon_only = n_colon,
    pct_conserved = round(100 * n_both / max(1, n_both + n_duod + n_colon), 1)
  )
}

if (length(summary_reg_rows) > 0) {
  reg_summary_df <- do.call(rbind, summary_reg_rows)
  write.csv(reg_summary_df,
            file.path(comp_dir_ent, "Enterocytes_Region_Comparison_Summary.csv"),
            row.names = FALSE)
  message("\nEnterocyte region comparison summary:")
  print(reg_summary_df)
}

Need results from at least 2 regions.



  No significant peaks for: Node1_Human_vs_Pan



Need results from at least 2 regions.



  No significant peaks for: Div_Human_vs_Apes



Need results from at least 2 regions.



...

  No significant peaks for: Node3_GreatApes_vs_Macaque



Need results from at least 2 regions.



  No significant peaks for: Node4_OldWorld_vs_Marmoset



Need results from at least 2 regions.



  No significant peaks for: Node_Apes_vs_Monkeys



In [17]:
# =============================================================================
# Cell 17: Per-Region Accessibility Dotplots for Enterocytes
# =============================================================================
# For each region, show the top human-specific peaks as a dotplot,
# so we can visually compare Duodenum vs Colon regulatory landscapes.

for (region in TARGET_REGIONS) {
  if (is.null(region_res_ent[[region]])) next
  if (is.null(region_res_ent[[region]][[ENT_CT]])) next

  # Focus on human divergence contrast
  cn <- "Div_Human_vs_AllPrimates"
  res_r <- region_res_ent[[region]][[ENT_CT]][[cn]]
  if (is.null(res_r)) next

  res_df_r <- as.data.frame(res_r)
  top_peaks_r <- rownames(res_df_r)[
    !is.na(res_df_r$padj) & res_df_r$padj < 0.05 &
    res_df_r$log2FoldChange > 1
  ]
  top_peaks_r <- head(
    top_peaks_r[order(res_df_r[top_peaks_r, "log2FoldChange"], decreasing = TRUE)],
    30
  )
  if (length(top_peaks_r) < 5) next

  reg_samp <- rownames(meta_ent_reg)[meta_ent_reg$region == region]
  tryCatch({
    plot_accessibility_dotplot(
      peak_ids  = top_peaks_r,
      counts    = counts_ent_u_reg[, reg_samp, drop = FALSE],
      meta      = meta_ent_reg[reg_samp, ],
      out_file  = file.path(ENT_PLOT,
                            paste0("Dotplot_", region, "_HumanSpecific.pdf")),
      title     = paste("Enterocytes –", region, "– Human-specific peaks")
    )
  }, error = function(e) message("  Dotplot failed for ", region, ": ", e$message))
}

message("Per-region enterocyte dotplots saved.")

  Accessibility dotplot saved: Dotplot_Duodenum_HumanSpecific.pdf



Per-region enterocyte dotplots saved.



In [18]:
# =============================================================================
# Cell 18: Region Comparison Heatmap for Enterocytes
# =============================================================================
tryCatch({
  plot_region_comparison_heatmap(
    region_res = region_res_ent,
    cell_type  = ENT_CT,
    out_file   = file.path(ENT_PLOT, "Region_Comparison_Heatmap_Enterocytes.pdf")
  )
}, error = function(e) {
  message("Region comparison heatmap failed: ", e$message)
})

`use_raster` is automatically set to TRUE for a matrix with more than
2000 rows. You can control `use_raster` argument by explicitly setting
TRUE/FALSE to it.

Set `ht_opt$message = FALSE` to turn off this message.



Region comparison heatmap failed: size cannot be NA nor exceed 65536



In [19]:
# =============================================================================
# Cell 19: Final Checkpoint & Output Summary
# =============================================================================
message("\n========== Enterocyte Deep-Dive Complete ==========")
message()
message("=== Level 1 — Focused DESeq2 ===")
message("  Shared-peak contrasts (all-vs-one): ", length(res_ent_shared[[ENT_CT]]))
message("  Evolutionary branch contrasts:      ", length(branch_ent[[ENT_CT]]))
total_ur <- sum(sapply(robust_ent[[ENT_CT]], length))
message("  Ultra-robust peaks (total):         ", total_ur)

message()
message("=== Level 2 — Motif Enrichment ===")
message("  Results in: ", file.path(ENT_OUT, "motif_enrichment"))

message()
message("=== Level 3 — Gene Annotation & Visualization ===")
message("  Annotated peaks: ",
        file.path(ENT_OUT, "gene_annotation", "Enterocytes_UltraRobust_Annotated.csv"))
message("  GO enrichment:   ", file.path(ENT_OUT, "differential_results", "enrichment"))
message("  All plots:       ", ENT_PLOT)

message()
message("=== Per-Region ===")
for (rg in TARGET_REGIONS) {
  if (!is.null(region_res_ent[[rg]])) {
    n_c <- length(region_res_ent[[rg]][[ENT_CT]])
    message(sprintf("  %-12s: %d contrast results", rg, n_c))
  }
}


========== Enterocyte Deep-Dive Complete ==========



=== Level 1 <U+2014> Focused DESeq2 ===



  Shared-peak contrasts (all-vs-one): 6



  Evolutionary branch contrasts:      19



  Ultra-robust peaks (total):         101559



...

  GO enrichment:   /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/differential_results/enterocytes/differential_results/enrichment



  All plots:       /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/plots/Enterocytes_Deep



=== Per-Region ===



  Duodenum    : 19 contrast results



  Colon       : 0 contrast results

